# သင်ခန်းစာ ၀၈ - Multi-Agent ဒီဇိုင်းပုံစံ


## ပြင်ဆင်မှု


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ဘာကြောင့် Multi-Agent Systems လဲ?

ခရီးစဉ် စီစဉ်ခြင်းကဲ့သို့သော မြေပြင်တွင် ရှိသော လုပ်ဆောင်ချက်များသည် အမျိုးမျိုးသော ကျွမ်းကျင်မှုများကို လိုအပ်သလို — ကုန်သွယ်မှု စီမံခန့်ခွဲမှု၊ ဒေသဆိုင်ရာ အသိပညာ၊ ငွေကြေး စီမံခန့်ခွဲမှု၊ နှင့် အခြားစသည်တို့ ပါဝင်သည်။ တစ်ဦးတည်းသော အေးဂျင့်တစ် ဦးက အားလုံးကို ကိုင်တွယ်ရန် ကြိုးစားသောအခါ နေရာတစ်ခုတည်းမှာ တတ်နိုင်သမျှ မထိရောက်တော့ပါ။

Multi-agent systems သည် **အထူးပြုကျွမ်းကျင်မှု** မှတဆင့် ဤပြဿနာကို ဖြေရှင်းသည်။ အေးဂျင့် တစ်ဦးချင်းစီမှာ တစ်ခုတည်းသော ကျွမ်းကျင်မှုထဲကို အာရုံစိုက်ရာမှ ဘယ်သူမဆို အနည်တူ ကျွမ်းကျင်သူထက် ပိုမိုအရည်အသွေးမြင့်သော ရလဒ်များ ထုတ်လုပ်နိုင်ပါသည်။ ၎င်းတို့သည် **တိုးချဲ့နိုင်မှု** ကိုလည်း တိုးတက်စေပါသည် — အစားထိုးပေးရမည့် လုပ်ငန်းစဉ်များကို ပြန်ရေးရန် မလိုဘဲ အေးဂျင့်အသစ်များ (ဥပမာ၊ လေကြောင်းအထူးကျွမ်းကျင်သူ၊ ဟိုတယ်သုံးသပ်သူ) ကို ထည့်သွင်းနိုင်ပါသည်။ အေးဂျင့်များသည် တိကျစွာ ဖွဲ့စည်းထားသော လမ်းကြောင်းမှတစ်ဆင့် ပုံစံသတ်မှတ်လုပ်ကာ တစ်ဦးမှတစ်ဦးသို့ အချက်အလက်များကို လွှဲပြောင်းပေးကြသည်။


## အထူးပြုထားသော အေးဂျင့်များ ဖန်တီးခြင်း


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## အဆက်အသွယ်ရှိ Workflow တစ်ခု တည်ဆောက်ခြင်း

`WorkflowBuilder` သည် စက်ရုပ်များကို ဦးတည်ချက်ရှိသော graph ထဲချိတ်ဆက်ရန် ခွင့်ပြုသည်။ ဒီမှာ ကျွန်တော်တို့ ရိုးရှင်းတဲ့ နှစ်ဆင့် pipeline တစ်ခု တည်ဆောက်မှာဖြစ်ပြီး- **TravelPlanner** က ခရီးစဉ် ကို မူကြမ်းရေးဆွဲပြီး၊ ထို့နောက် **TravelConcierge** က စစ်ဆေးပြီး တိုးတက်အောင် ပြုစုပေးသည်။


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Workflow သို့ ကိုယ်စားလှယ်များ ပိုထည့်ခြင်း

multi-agent ပုံစံရဲ့ အကြီးဆုံးအားသာချက် တစ်ခုကတော့ တိုးချဲ့ဖို့ အလွန် လွယ်ကူတဲ့ အချက်ဖြစ်ပါတယ်။ အောက်တွင် **BudgetReviewer** ကိုယ်စားလှယ်တစ် ဦး ကိုထည့်သွင်းထားပြီး၊ ခရီးသွားသူရဲ့ စည်စဥ်ပေါ်ခံကိုစစ်ဆေးပြီး၊ ကုန်ကျစရိတ် ပြည့်ကျော်နိုင်တဲ့ အချက်များကိုမှတ်သားပေးပြီး၊ ပိုမို ငွေမျှတစေမည့် အစားထိုးဆွေးနွေးချက်များကို အကြံပြုသည်။ ဒီ workflow က အခုတော့ ကိုယ်စားလှယ်သုံးယောက်ကို ဆက်တိုက် ပြေးဆွဲနေပါပြီ။

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

ဤသင်ခန်းစာတွင် သင်သည် အောက်ပါအတိုင်း ကိုင်တွယ်နိုင်သည်-

1. **အထူးပြု ကိရိယာများ ဖန်တီးခြင်း** — တစ်ဦးချင်းစီမှာ ဂရုစိုက်ထားတဲ့ အခန်းကဏ္ဍ (အစီအစဉ်ရေးဆွဲမှု, ဝန်ဆောင်မှု, ဘတ်ဂျက် သုံးသပ်မှု)
2. `WorkflowBuilder` နှင့် `add_edge` ကိုအသုံးပြုပြီး ကိရိယာများကို ချိတ်ဆက်ထားသော အဆင့်လိုက် ဝန်းရံဖြစ်စေခြင်း
3. များစွာသော ကိရိယာထဲကနေ ထွက်ရှိလာသော output ကို မကြာခဏထွက်မြောက်စေခြင်း၊ ဘယ် ကိရိယာက ပြောနေသည်ကို သွယ်ဝိုက်လုပ်ဆောင်ခြင်း
4. ရှိပြီးသား ကိရိယာများကို မပြောင်းလဲဘဲ ဝန်းရံလုပ်ငန်းစဉ်ထဲက နယူး ကိရိယာအသစ်များ ထပ်တိုးထည့်ခြင်း

Multi-agent ဒီဇိုင်းနမူနာသည် ကိရိယာတစ်ခုချင်းစီကို ရိုးရှင်းစေပြီး၊ တစ်ဦးတည်းကဏ္ဍမှမရနိုင်သည့် ပိုမိုစုံလင်ပြီး သေချာစွာသုံးသပ်ပြီးသော ရလဒ်များကို ထုတ်လုပ်နိုင်စေသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ခြွင်းချက်**  
ဤစာရွက်စာတမ်းကို AI ဘာသာပြန်စနစ် [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းထားသော်လည်း၊ အလိုအလျောက် ဘာသာပြန်မှုများတွင် အမှားများ သို့မဟုတ် မှားယွင်းမှုများ ပါဝင်နိုင်ကြောင်း သိရှိရန် ကျေးဇူးပြု၍ သတိပြုပါ။ မူလစာရွက်စာတမ်းကို သူမည့်ဘာသာဖြင့်ရေးသားထားမှသာ အတိအကျ အချက်အလက် များအဖြစ် ယူဆရမည်ဖြစ်သည်။ အရေးကြီးသော အချက်အလက်များအတွက် လူ့專業 ဘာသာပြန်သူ၏ တိကျမှန်ကန်မှုရှိသော ဘာသာပြန်မှုကို အသုံးပြုရန် အကြံပြုပါသည်။ ဤဘာသာပြန်မှုကို အသုံးပြုခြင်းကြောင့် ဖြစ်ပေါ်လာနိုင်သည့် နားလည်မှုအမှားများ သို့မဟုတ် မွန်းကြပ်မှုများအတွက် ကျွန်ုပ်တို့သည် တာဝန် မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
